# Convolutional neural networks

In this notebook we will train some deep neural networks based on geometrical features. In particular, we will train a CNN, because it is supposed to be suitable for timeseries datasets.

Scikitlearn does not allow convolutional neural networks, so we will use Tensorflow.

## Imports

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import classification_report, confusion_matrix

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "src" / "data_preprocessing.py").is_file()
     and (path / "notebooks").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Start Jupyter from the project folder or a notebook subfolder.")

sys.path.insert(0, str(PROJECT_ROOT / "src"))
from data_preprocessing import group_aami_classes

keras.utils.set_random_seed(42)

Let's load data

In [ ]:
data_dir = PROJECT_ROOT / "data"
if not all((data_dir / name).is_file() for name in (
    "train_data.csv", "val_data.csv", "test_data.csv"
)):
    raise FileNotFoundError("Dataset missing: see data/README.md for the required CSV files.")


train_data = pd.read_csv(data_dir / "train_data.csv", header=None)
val_data = pd.read_csv(data_dir / "val_data.csv", header=None)
test_data = pd.read_csv(data_dir / "test_data.csv", header=None)

print("Training set shape:", train_data.shape)
print("Validation set shape:", val_data.shape)
print("Test set shape:", test_data.shape)

train_data.head()

We have to split data and group them into the 5 classes.

In [ ]:
X_train = train_data.iloc[:, :-1].to_numpy()
y_train = train_data.iloc[:, -1].astype(int).to_numpy()

X_val = val_data.iloc[:, :-1].to_numpy()
y_val = val_data.iloc[:, -1].astype(int).to_numpy()

X_test = test_data.iloc[:, :-1].to_numpy()
y_test = test_data.iloc[:, -1].astype(int).to_numpy()

y_train = group_aami_classes(y_train)
y_val = group_aami_classes(y_val)
y_test = group_aami_classes(y_test)

## CNN training

First, we need to prepare data for the CNN. 1 means that we consider one channel per ECG. This is done because in order to train the CNN we need to specify how many channels (measurements) we have for each initial feature for each sample. 

In [ ]:
# With these lines we convert the matrices into 3D tensors. The cast into float32 is done
# because it is the common used type for Keras.

X_train_cnn = X_train.astype("float32").reshape(-1, 250, 1)
X_val_cnn = X_val.astype("float32").reshape(-1, 250, 1)
X_test_cnn = X_test.astype("float32").reshape(-1, 250, 1)

We are ready to build the network.

In [ ]:
# In Keras we need to specify the type of each layer of the network.

cnn_raw = keras.Sequential([
    # The first layer is the input one, that is (250,1)
    keras.Input(shape=(250, 1)),
    # The first actual layer is the convolutional one. 1D, with 16 filters each one
    # with 5 weights and one bias. The output is (n_samples, 250, 16)
    layers.Conv1D(
        filters=16,
        kernel_size=5,
        # With padding = "same" we keep the same inizial output dimension via padding.
        # So, this layer will have 250 outputs.
        padding="same",
        # After the convolution, the network will apply the ReLU (max(0,x)=ReLU(x)).
        activation="relu",
    ),
    # The Convolutional layerl will produce 16 channels, each one with 250 elements.

    # Now we can add a MaxPooling layer, that reduces the dimension of the input. 
    layers.MaxPooling1D(pool_size=2),

    # Now we can add another convolutional layer to learn more complex patterns (5x16 filter size).
    layers.Conv1D(
        filters=32,
        kernel_size=5,
        padding="same",
        activation="relu",
    ),
    # Another MaxPooling layer. Now the output is (62x32).
    layers.MaxPooling1D(pool_size=2),

    # We add a flattener layer. The output is (n_samples, 1984).
    layers.Flatten(),
    # Now there is the hidden layer of a fully connected network. It has 64 neurons and only one layer.
    layers.Dense(64, activation="relu"),
    # Lastly, the output layer with 5 softmax neurons.
    layers.Dense(5, activation="softmax"),
])

In [ ]:
# In Keras we need to specify the optimizer in a different function. 
cnn_raw.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

We use Adam with learning rate 0.001 and sparse categorical
cross-entropy, since class labels are integers. Training runs for
at most 50 epochs with batches of 128 samples.

Early stopping monitors validation loss, stops after five epochs
without improvement and restores the weights with the lowest
validation loss. 

In [ ]:
# We can add an early stopping setting that stops the training when the validation error stop decreasing.
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1,
)

# Finally we can train the network.
cnn_raw.fit(
    X_train_cnn,
    y_train,
    validation_data=(X_val_cnn, y_val),
    epochs=50,
    batch_size=128,
    shuffle=True,
    callbacks=[early_stopping],
    verbose=2,
)

In [ ]:
# Usual print
y_val_prob = cnn_raw.predict(X_val_cnn, verbose=0)
y_val_pred = np.argmax(y_val_prob, axis=1)

print("Validation results:")
print(classification_report(
    y_val,
    y_val_pred,
    labels=[0, 1, 2, 3, 4],
    target_names=["N", "S", "V", "F", "Q"],
    digits=4,
    zero_division=0,
))

print("Confusion matrix (rows: true classes, columns: predictions):")
print(confusion_matrix(
    y_val, y_val_pred, labels=[0, 1, 2, 3, 4]
))

Let's try to add some dropout.

In [ ]:
keras.utils.set_random_seed(42)

cnn_dropout = keras.Sequential([
    keras.Input(shape=(250, 1)),

    layers.Conv1D(
        filters=16, kernel_size=5, padding="same", activation="relu"
    ),
    layers.MaxPooling1D(pool_size=2),

    layers.Conv1D(
        filters=32, kernel_size=5, padding="same", activation="relu"
    ),
    layers.MaxPooling1D(pool_size=2),

    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(5, activation="softmax"),
])

cnn_dropout.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

early_stopping_dropout = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1,
)

cnn_dropout.fit(
    X_train_cnn,
    y_train,
    validation_data=(X_val_cnn, y_val),
    epochs=50,
    batch_size=128,
    shuffle=True,
    callbacks=[early_stopping_dropout],
    verbose=2,
)

In [ ]:
y_val_prob = cnn_dropout.predict(X_val_cnn, verbose=0)
y_val_pred = np.argmax(y_val_prob, axis=1)

print("Validation results:")
print(classification_report(
    y_val,
    y_val_pred,
    labels=[0, 1, 2, 3, 4],
    target_names=["N", "S", "V", "F", "Q"],
    digits=4,
    zero_division=0,
))

print("Confusion matrix (rows: true classes, columns: predictions):")
print(confusion_matrix(
    y_val, y_val_pred, labels=[0, 1, 2, 3, 4]
))

The result are almost the same as before.
Let's try to add another dense layer without dropout.

In [ ]:

keras.utils.set_random_seed(42)

cnn_2_layers = keras.Sequential([
    keras.Input(shape=(250, 1)),

    layers.Conv1D(
        filters=16, kernel_size=5, padding="same", activation="relu"
    ),
    layers.MaxPooling1D(pool_size=2),

    layers.Conv1D(
        filters=32, kernel_size=5, padding="same", activation="relu"
    ),
    layers.MaxPooling1D(pool_size=2),

    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dense(64, activation="relu"),
    layers.Dense(5, activation="softmax"),
])

cnn_2_layers.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

early_stopping_2_layers = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1,
)

cnn_2_layers.fit(
    X_train_cnn,
    y_train,
    validation_data=(X_val_cnn, y_val),
    epochs=50,
    batch_size=128,
    shuffle=True,
    callbacks=[early_stopping_2_layers],
    verbose=2,
)

In [ ]:
y_val_prob = cnn_2_layers.predict(X_val_cnn, verbose=0)
y_val_pred = np.argmax(y_val_prob, axis=1)

print("Validation results:")
print(classification_report(
    y_val,
    y_val_pred,
    labels=[0, 1, 2, 3, 4],
    target_names=["N", "S", "V", "F", "Q"],
    digits=4,
    zero_division=0,
))

print("Confusion matrix (rows: true classes, columns: predictions):")
print(confusion_matrix(
    y_val, y_val_pred, labels=[0, 1, 2, 3, 4]))

Validation macro F1 is 0.9251 for the baseline CNN, 0.9273 with
dropout and 0.9178 with two dense hidden layers. We therefore
select the dropout configuration for final test evaluation.

## Final test evaluation and conclusions

We evaluate the CNN selected using the validation set on the test set.
The selected model includes a dropout layer with rate 0.3.

In [ ]:
y_test_prob = cnn_dropout.predict(X_test_cnn, verbose=0)
y_test_pred = np.argmax(y_test_prob, axis=1)

print("Test results:")
print(classification_report(
    y_test,
    y_test_pred,
    labels=[0, 1, 2, 3, 4],
    target_names=["N", "S", "V", "F", "Q"],
    digits=4,
    zero_division=0,
))

print("Confusion matrix (rows: true classes, columns: predictions):")
print(confusion_matrix(
    y_test,
    y_test_pred,
    labels=[0, 1, 2, 3, 4],
))

Its test macro F1 is 0.9134. Among the configurations tested,
the CNN does not exceed the raw-data MLP's validation macro F1
of 0.9375.